# Day 17: Introduction to PyTorch

**PyTorch** is an open-source deep learning library developed by Meta AI. It is used to build and train neural networks.

### Key ideas
| Concept | What it is |
|---|---|
| **Tensor** | A multi-dimensional array, similar to a NumPy array, that can also run on a GPU. All data and model weights are tensors. |
| **Autograd** | PyTorch's automatic differentiation engine. It records the operations you run and computes gradients for you during training. |
| **`torch.nn`** | The module with ready-made layers (`nn.Linear`), activations (`nn.ReLU`) and loss functions (`nn.MSELoss`). |
| **`nn.Module`** | The base class that every PyTorch model inherits from. |

### What this notebook covers
1. Installing PyTorch
2. Checking the version and whether a GPU is available
3. Defining a simple feed-forward neural network (`CustomerPredictor`)
4. Tensors: what they are and why a neural network needs them (with a sample input)
5. Inspecting the model's parameters (weights and biases)

## 1. Installation

```
pip install torch
```

- In a notebook, a leading `!` runs the line as a shell command (`!pip install torch`).
- The default `pip install torch` on Windows installs the **CPU-only** build. For NVIDIA GPU support, use the CUDA install command from [pytorch.org/get-started](https://pytorch.org/get-started/locally/).
- **Restart the kernel** after installing so the notebook picks up the new package.
- The line below is commented out because PyTorch is already installed. Uncomment it only on a new environment.

In [180]:
# !pip install torch

## 2. Checking the Installation

- `torch.__version__` shows the installed PyTorch version.
  - The suffix tells you which build is installed. `+cpu` means a **CPU-only** build. A build such as `+cu121` includes CUDA 12.1 support for NVIDIA GPUs.
- `torch.cuda.is_available()` returns `True` only when **both** of these are true:
  1. An NVIDIA GPU with a working driver is present.
  2. A CUDA-enabled build of PyTorch is installed.

This machine prints `2.14.1+cpu` and `False`, so everything runs on the CPU. That is fine for small models like the one below.

A common pattern lets the same code run on either device:
```python
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)   # the model and its data must be on the same device
```

In [181]:
import torch


# Print the installed PyTorch version
print("PyTorch Version:", torch.__version__)

print("GPU Available:", torch.cuda.is_available())


PyTorch Version: 2.14.1+cpu
GPU Available: False


## 3. Building a Neural Network with `nn.Module`

`torch.nn` contains the building blocks for neural networks: layers, activation functions and loss functions.

### How a PyTorch model is defined
Every model is a class that **inherits from `nn.Module`** and has two parts:

| Part | Purpose |
|---|---|
| `__init__` | Create the layers (the parts with learnable weights). |
| `forward(x)` | Describe how the input moves through those layers to give an output. |

- `super().__init__()` must be called first. It sets up `nn.Module` so PyTorch can track every layer and its parameters (`model.parameters()`).
- The model is run as `model(x)`, **not** `model.forward(x)`. Calling the model runs `forward` plus PyTorch's internal hooks.
- You never write the backward pass. **Autograd** works out the gradients from `forward`.

### `nn.Linear(in_features, out_features)`
This is a fully connected (dense) layer. Every input is connected to every output neuron:

$$y = xW^T + b$$

- `weight` has shape `(out_features, in_features)`
- `bias` has shape `(out_features,)`
- Number of parameters = `in_features × out_features + out_features`

### Architecture of `CustomerPredictor`

![CustomerPredictor network: 3 input features, a hidden layer of 8 neurons with ReLU, and 1 output, all fully connected](images/customer_predictor_network.svg)

Each line in the diagram is one **weight**. Every node in the hidden and output layers also has one **bias**.

| Layer | Shape | Weights | Biases | Parameters |
|---|---|---|---|---|
| `hidden` | 3 → 8 | 3 × 8 = 24 | 8 | 32 |
| `output` | 8 → 1 | 8 × 1 = 8 | 1 | 9 |
| **Total** | | | | **41** |

This network has **one hidden layer with 8 neurons**. It is not 8 hidden layers. To add more hidden layers, add more `nn.Linear` layers in `__init__` and pass `x` through each of them in `forward`.

### ReLU activation
$$\text{ReLU}(x) = \max(0, x)$$

- Negative values become 0 and positive values pass through unchanged.
- It adds **non-linearity**. Without an activation, two `Linear` layers in a row are the same as one `Linear` layer, so the network could only learn straight-line (linear) relationships.
- ReLU is the default choice for hidden layers because it is cheap to compute and helps avoid vanishing gradients.

### Why there is no activation on the output layer
The output layer returns a **raw score (logit)**. The loss function applies the right final step itself:

| Task | Output size | Loss function | Final step applied by the loss |
|---|---|---|---|
| Regression (predict a number, e.g. spend) | 1 | `nn.MSELoss` | none |
| Binary classification (e.g. churn yes/no) | 1 | `nn.BCEWithLogitsLoss` | sigmoid |
| Multi-class classification (N classes) | N | `nn.CrossEntropyLoss` | softmax |

> **Note:** `nn.Linear(8, 1)` gives a **single output**, so this model fits regression or binary (yes/no) classification. For N classes, you would use `nn.Linear(8, N)` with `nn.CrossEntropyLoss`.

### The forward pass, step by step

| Step | Code | What happens | Shape |
|---|---|---|---|
| 1 | `logits = self.hidden(x)` | Linear transformation $z = xW^T + b$. Each of the 8 hidden neurons takes a weighted sum of the 3 inputs and adds its bias. The results are raw values and can be negative. | `(batch, 3)` → `(batch, 8)` |
| 2 | `x = torch.relu(logits)` | ReLU turns negative values into 0 and keeps positive values unchanged. | `(batch, 8)` → `(batch, 8)` |
| 3 | `return self.output(x)` | The output layer combines the 8 activated values into one raw score. | `(batch, 8)` → `(batch, 1)` |

> **Naming:** *logits* usually means the raw output of the **final** layer, the value that goes into sigmoid or softmax. The hidden layer's values before ReLU are more often called `z` or *pre-activations*. The code works the same either way.

### Creating the model
`model = CustomerPredictor()` runs `__init__`, which creates both layers. Their weights and biases start as **small random values** (PyTorch's default initialisation), so the model's predictions mean nothing until it is trained.

Useful checks:
```python
print(model)                                  # summary of the layers
sum(p.numel() for p in model.parameters())    # 41 trainable parameters
```

In [182]:
from torch import nn


class CustomerPredictor(nn.Module):
    def __init__(self):
        super().__init__()  # initialise nn.Module so the layers below get registered
        # input -> hidden: 3 input features, 8 neurons in one hidden layer
        self.hidden = nn.Linear(3, 8)
        # hidden -> output: 8 neurons in, 1 output value
        self.output = nn.Linear(8, 1)

    # forward pass: defines how an input flows through the network
    def forward(self, x):
        # Step 1: hidden layer applies the linear transformation z = x @ W.T + b
        # x has shape (batch, 3) -> logits has shape (batch, 8), one value per hidden neuron.
        # These are raw (pre-activation) values: any real number, positive or negative.
        logits = self.hidden(x)

        # Step 2: ReLU keeps positive values and turns negative values into 0 -> max(0, z).
        # This adds non-linearity, so the network can learn more than straight-line patterns.
        x = torch.relu(logits)

        # Step 3: output layer combines the 8 activated values into 1 raw score, shape (batch, 1).
        return self.output(x)


# Create an instance of the model
model = CustomerPredictor()

## 4. Tensors: the Input to a Neural Network

### Q1. What is a tensor?

A **tensor** is PyTorch's container for numbers: a grid of values laid out along one or more **axes** (dimensions). The number of axes is the tensor's **rank**.

![Tensor ranks: a scalar, a vector, a matrix and a 3-D tensor, with their shapes](images/tensor_dimensions.svg)

| Rank | Name | Example code | Shape | Real-world example |
|---|---|---|---|---|
| 0 | Scalar | `torch.tensor(5)` | `[]` | a single price |
| 1 | Vector | `torch.tensor([0.5, 1.0, 2.0])` | `[3]` | one customer's features |
| 2 | Matrix | `torch.tensor([[0.5, 1.0, 2.0]])` | `[1, 3]` | customers (rows) × features (columns) |
| 3 | 3-D tensor | `torch.rand(3, 28, 28)` | `[3, 28, 28]` | a colour image: channels × height × width |

Every tensor has three attributes you will check often:

| Attribute | Meaning | Value for `sample_tensor` |
|---|---|---|
| `.shape` | the size of each axis | `torch.Size([1, 3])` |
| `.dtype` | the type of the numbers | `torch.float32` |
| `.device` | where it is stored | `cpu` (or `cuda` on a GPU) |

#### Tensor vs NumPy array vs Python list

| Feature | Python list | NumPy array | PyTorch tensor |
|---|:---:|:---:|:---:|
| Fast maths on the whole grid | ❌ | ✅ | ✅ |
| One number type (dtype) for all values | ❌ | ✅ | ✅ |
| Runs on a GPU | ❌ | ❌ | ✅ |
| Tracks gradients (autograd) | ❌ | ❌ | ✅ |

> **In one line:** a tensor is like a NumPy array that can also run on a GPU and remember the maths done on it, so PyTorch can compute gradients.

Converting between them: `torch.tensor(my_list)`, `torch.from_numpy(my_array)` and `my_tensor.numpy()`.

### Q2. Why does a neural network expect tensors as input?

A neural network is a **chain of maths operations on tensors**. Its weights and biases are tensors, so the input must also be a tensor with the right **shape**, number type (**dtype**) and **device**.

![How a tensor flows through CustomerPredictor: customer data, input tensor, hidden layer with ReLU, output, and the training loop](images/tensor_flow.svg)

The numbered badges in the diagram match these four reasons:

| # | Reason | What it means for `CustomerPredictor` |
|---|---|---|
| ① | **Numbers only** | A network can only do arithmetic. Raw data (age, visits, spend) is turned into numbers, usually scaled to small values like `0.5, 1.0, 2.0`, and stored in a tensor. |
| ② | **Matrix maths** | Each `nn.Linear` layer computes `x @ Wᵀ + b`. `W` and `b` are `float32` tensors, so `x` must be a `float32` tensor with a matching shape. |
| ③ | **Batches** | A 2-D tensor holds many samples as rows. One matrix multiplication handles the whole batch at once, which is much faster than a Python loop. |
| ④ | **Autograd + GPU** | Tensors record the operations done on them, so `loss.backward()` can compute a gradient for every weight. Tensors can also be moved to a GPU with `.to("cuda")`. |

#### Inside the hidden layer: one matrix multiplication

![Matrix multiplication inside the hidden layer: x (1, 3) times W transpose (3, 8) plus b (8) gives z (1, 8)](images/tensor_matmul.svg)

- `x` has shape `(1, 3)` and `Wᵀ` has shape `(3, 8)`. The **inner sizes (3 and 3) must match**. The result takes the **outer sizes**: `(1, 8)`.
- Each hidden neuron takes a **weighted sum of all 3 inputs plus its own bias**. That is why the input's last dimension must equal `in_features` in `nn.Linear(3, 8)`.

#### What goes wrong without the right tensor

| Input | Result |
|---|---|
| `model([0.5, 1.0, 2.0])`: a Python list | ❌ `TypeError: linear(): argument 'input' (position 1) must be Tensor, not list` |
| `model(torch.tensor([[1, 2, 3]]))`: integers (`int64`) | ❌ `RuntimeError: mat1 and mat2 must have the same dtype, but got Long and Float` |
| `model(torch.tensor([[0.5, 1.0]]))`: only 2 features | ❌ `RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x2 and 3x8)` |
| `model(torch.tensor([[0.5, 1.0, 2.0]]))` | ✅ returns a tensor with shape `(1, 1)` |

### The sample tensor in this notebook

```python
sample_tensor = torch.tensor([[0.5, 1.0, 2.0]])
```

| What to notice | Why it matters |
|---|---|
| **Double brackets** `[[ ... ]]` | They make a 2-D tensor with shape `(1, 3)`: **1 row** (one customer) × **3 columns** (three features). |
| **Batch dimension first** | Layers expect `(batch_size, features)`. For more customers, add more rows. For example, `[[0.5, 1.0, 2.0], [1.5, 0.2, 3.1]]` gives shape `(2, 3)`. |
| **Last dimension = 3** | It must equal `in_features` of the first layer, `nn.Linear(3, 8)`. |
| **Decimal numbers** | They give `torch.float32`, the same type as the weights. For whole numbers, add `dtype=torch.float32`. |

Passing it through the model:

```
sample_tensor (1, 3) ──► hidden (1, 8) ──► ReLU (1, 8) ──► output (1, 1)
```

`model(sample_tensor)` returns **one raw score** with shape `(1, 1)`. The weights are still random, so the value has no meaning until the model is trained.

In [183]:
print(list(model.parameters()))

[Parameter containing:
tensor([[-0.0584, -0.5485,  0.0976],
        [-0.0704,  0.5328,  0.1431],
        [-0.2386,  0.2494, -0.2701],
        [-0.4605,  0.2444,  0.1035],
        [-0.4679, -0.1585,  0.4110],
        [-0.3328, -0.2831,  0.1187],
        [-0.0301,  0.1149,  0.1528],
        [-0.3160, -0.3289, -0.2491]], requires_grad=True), Parameter containing:
tensor([-0.4066, -0.4611, -0.0243,  0.0022,  0.2182, -0.1408, -0.3891, -0.0826],
       requires_grad=True), Parameter containing:
tensor([[ 0.2863, -0.1891,  0.1242, -0.0154, -0.1678,  0.2428,  0.2462, -0.2198]],
       requires_grad=True), Parameter containing:
tensor([0.0483], requires_grad=True)]


## 5. Inspecting the Model's Parameters

`model.parameters()` gives every **learnable tensor** in the model: the weights and biases that training adjusts. It returns a *generator*, so we wrap it in `list(...)` to print everything at once.

### Reading the output of `print(list(model.parameters()))`

![The four tensors printed by model.parameters(), mapped onto the hidden and output layers of CustomerPredictor](images/model_parameters.svg)

The list holds **4 tensors**, in the same order as the layers are created in `__init__`:

| # | What is printed | Name | Shape | What each number is |
|---|---|---|---|---|
| ① | 8 rows × 3 numbers | `hidden.weight` | `(8, 3)` | Row *i* holds the 3 weights going **into** hidden neuron hᵢ, one from each input x₁, x₂, x₃ |
| ② | 8 numbers | `hidden.bias` | `(8,)` | One bias for each hidden neuron |
| ③ | 1 row × 8 numbers | `output.weight` | `(1, 8)` | The 8 weights going into the output, one from each hidden neuron |
| ④ | 1 number | `output.bias` | `(1,)` | The output neuron's bias |
| | | **Total** | | 24 + 8 + 8 + 1 = **41** |

#### The parts of each printed item

| Printed text | Meaning |
|---|---|
| `Parameter containing:` | The tensor is an `nn.Parameter`: a tensor that `nn.Module` registers as learnable. That is why it appears in `model.parameters()`, and why the optimizer will update it. |
| `tensor([[ ... ]])` | The values. Double brackets mean 2-D (a weight matrix). Single brackets mean 1-D (a bias vector). |
| `requires_grad=True` | Autograd will compute a gradient for this tensor when `loss.backward()` runs. |

#### Why is the weight shape `(8, 3)` and not `(3, 8)`?
PyTorch stores weights as `(out_features, in_features)`, so each neuron gets **one row**. The forward pass uses the transpose, `x @ W.T + b`. That transpose is the `Wᵀ (3, 8)` grid in the Section 4 matrix diagram.

#### Following `sample_tensor` through these exact weights

**Hidden neuron h₁** uses row 1 of ① and value 1 of ②:

| Input | Value | × Weight | = |
|---|---|---|---|
| x₁ | 0.5 | -0.3095 | -0.15475 |
| x₂ | 1.0 | -0.0098 | -0.0098 |
| x₃ | 2.0 | 0.3581 | 0.7162 |
| + bias | | | -0.4767 |
| **z₁** | | | **0.07495** |

z₁ is positive, so `ReLU(z₁) = 0.07495`.

All 8 hidden neurons, done the same way (3 decimals):

| | h₁ | h₂ | h₃ | h₄ | h₅ | h₆ | h₇ | h₈ |
|---|---|---|---|---|---|---|---|---|
| z (before ReLU) | 0.075 | 0.215 | 0.381 | -1.031 | -0.938 | -0.932 | -0.201 | 1.134 |
| ReLU(z) | 0.075 | 0.215 | 0.381 | **0** | **0** | **0** | **0** | 1.134 |

ReLU switches off h₄ to h₇ for this customer. The output layer then uses ③ and ④:

```
ŷ = 0.075·(-0.1374) + 0.215·0.0876 + 0.381·0.2593 + 1.134·0.0449 + (-0.2337) ≈ -0.075
```

The printed weights are rounded to 4 decimals, so `model(sample_tensor)` can differ slightly in the last digit.

#### Why are the numbers small?
`nn.Linear` starts with random values between **±1/√in_features**:

| Layer | in_features | Starting range | Printed values |
|---|---|---|---|
| hidden (① ②) | 3 | ±0.577 | all inside the range (largest: 0.5743) |
| output (③ ④) | 8 | ±0.354 | all inside the range (largest: 0.2593) |

Small starting values keep the first outputs stable. Because the values are random, each neuron starts out different, so each one can learn a different pattern.

> **Your numbers will change** every time you run `model = CustomerPredictor()`, because the starting weights are random. To get the same numbers every run, call `torch.manual_seed(42)` before creating the model.

### All 41 numbers on the network

![CustomerPredictor network with every value from print(list(model.parameters())) placed on its connection line or beside its neuron](images/model_parameters_network.svg)

This is the same network as the Section 3 diagram, with each printed number placed where it is used:

| Printed tensor | Where it is on the diagram |
|---|---|
| ① `hidden.weight` (8 × 3) | The groups of 3 numbers left of h₁–h₈. **Row i is the group beside hᵢ.** In each group, the top, middle and bottom numbers are the lines from x₁, x₂ and x₃ (columns 1, 2, 3 of the printed row). |
| ② `hidden.bias` (8) | The number to the right of each hidden neuron, from h₁ down to h₈. |
| ③ `output.weight` (1 × 8) | The numbers on the 8 lines into ŷ, from h₁ (top) down to h₈ (bottom). |
| ④ `output.bias` (1) | The number beside ŷ. |

**Matching a printed number to its line:** in Python (counting from 0), `hidden.weight[i, j]` is the line **from input j+1 into hidden neuron i+1**:

```python
model.hidden.weight[0, 2]   # 0.3581 -> the line from x₃ into h₁
model.output.weight[0, 7]   # 0.0449 -> the line from h₈ into ŷ
```

In [ ]:
for name, param in model.named_parameters():
    print(name)
    print(param)
    print("Shape:", param.shape)
    print()

hidden.weight
Parameter containing:
tensor([[-0.0584, -0.5485,  0.0976],
        [-0.0704,  0.5328,  0.1431],
        [-0.2386,  0.2494, -0.2701],
        [-0.4605,  0.2444,  0.1035],
        [-0.4679, -0.1585,  0.4110],
        [-0.3328, -0.2831,  0.1187],
        [-0.0301,  0.1149,  0.1528],
        [-0.3160, -0.3289, -0.2491]], requires_grad=True)
Shape: torch.Size([8, 3])

hidden.bias
Parameter containing:
tensor([-0.4066, -0.4611, -0.0243,  0.0022,  0.2182, -0.1408, -0.3891, -0.0826],
       requires_grad=True)
Shape: torch.Size([8])

output.weight
Parameter containing:
tensor([[ 0.2863, -0.1891,  0.1242, -0.0154, -0.1678,  0.2428,  0.2462, -0.2198]],
       requires_grad=True)
Shape: torch.Size([1, 8])

output.bias
Parameter containing:
tensor([0.0483], requires_grad=True)
Shape: torch.Size([1])



### `named_parameters()`: the same tensors, with names

`model.named_parameters()` gives `(name, tensor)` pairs, so you can see which layer each tensor belongs to. Each name is built as **`<attribute name in __init__>.<weight or bias>`**:

| Printed name | Comes from | Printed shape | In the diagram |
|---|---|---|---|
| `hidden.weight` | `self.hidden = nn.Linear(3, 8)` | `torch.Size([8, 3])` | ① |
| `hidden.bias` | `self.hidden` | `torch.Size([8])` | ② |
| `output.weight` | `self.output = nn.Linear(8, 1)` | `torch.Size([1, 8])` | ③ |
| `output.bias` | `self.output` | `torch.Size([1])` | ④ |

- The values are **exactly the same** as in `print(list(model.parameters()))`. These are the same tensors, just labelled.
- `torch.Size([8])` is how PyTorch prints the shape `(8,)`: a 1-D tensor with 8 values.
- Renaming `self.hidden` to `self.layer1` in `__init__` would change the printed names to `layer1.weight` and `layer1.bias`.

In [186]:
sample_tensor = torch.tensor([[1, 2.5, 3.1]])  # Example input
model.eval()  # Set the model to evaluation mode
with torch.no_grad():  # Disable gradient calculation for inference
    output = model(sample_tensor)
    print("Model Output:", output)


Model Output: tensor([[-0.2154]])
